### Data Classes in Python (Guide)

[Article](https://realpython.com/python-data-classes/)

[Quiz](https://realpython.com/quizzes/python-data-classes/)

A data class is a **regular Python class**. The only thing that sets it apart is that it has basic data model methods like `.__init__()`, .`__repr__()`, and `.__eq__()` implemented for you.

 Use `@dataclass` to generate `.__init__()`, `.__repr__()`, and `.__eq__()` automatically. Dataclasses allow you to create classes quickly, but you can also add defaults, custom methods, ordering, immutability, inheritance, and even slots.

A data class is a class typically containing **mainly data**, although there **aren’t** really any restrictions. You create it using the `@dataclass` decorator, as follows:

In [1]:
from dataclasses import dataclass

@dataclass
class DataClassCard:
    rank: str
    suit: str

In [2]:
queen_of_hearts = DataClassCard('Q', 'Hearts')

In [3]:
queen_of_hearts.rank

'Q'

In [4]:
queen_of_hearts

DataClassCard(rank='Q', suit='Hearts')

In [5]:
queen_of_hearts == DataClassCard('Q', 'Hearts')

True

you can **instantiate**, **print**, and **compare** data class instances straight out of the box.

Compare that to a regular class:

In [6]:
class RegularCard:
    def __init__(self, rank, suit):
        self.rank = rank
        self.suit = suit

if you try to use this plain class, you’ll notice that the representation of the objects is not very descriptive, and for some reason a queen of hearts is not the same as a queen of hearts:

In [7]:
queen_of_hearts = RegularCard('Q', 'Hearts')

In [8]:
queen_of_hearts.rank

'Q'

In [10]:
queen_of_hearts

In [11]:
queen_of_hearts == RegularCard('Q', 'Hearts')

False

By default, data classes implement a `.__repr__()` method to provide a nice string representation and an `.__eq__()` method that can do basic object comparisons. For the `RegularCard` class to imitate the data class above, you need to add these methods as well:

In [13]:
class RegularCard:
    def __init__(self, rank, suit):
        self.rank = rank
        self.suit = suit

    def __repr__(self):
        return (f'{self.__class__.__name__}'
                f'(rank={self.rank!r}, suit={self.suit!r})')

    def __eq__(self, other):
        if other.__class__ is not self.__class__:
            return NotImplemented
        return (self.rank, self.suit) == (other.rank, other.suit)

you’ll see:

- How to add default values to data class fields
- How data classes allow for ordering of objects
- How to represent immutable data
- How data classes handle inheritance

### Alternatives to Data Classes

In [17]:
queen_of_hearts_tuple = ('Q', 'Hearts')

In [15]:
queen_of_hearts_dict = {'rank': 'Q', 'suit': 'Hearts'}

It works. However, it puts a lot of responsibility on you as a programmer.

Furthermore, using these structures is not ideal:

In [18]:
queen_of_hearts_tuple[0]  # No named access

'Q'

In [19]:
queen_of_hearts_dict['suit']  # Would be nicer with .suit

'Hearts'

A better alternative is the `namedtuple`. It has long been used to create readable small data structures.

We can in fact recreate the data class example above using a `namedtuple` like this:

In [1]:
from collections import namedtuple

NamedTupleCard = namedtuple('NamedTupleCard', ['rank', 'suit'])

This definition of `NamedTupleCard` will give the exact same output as our `DataClassCard` example did:

In [2]:
queen_of_hearts = NamedTupleCard('Q', 'Hearts')

In [22]:
queen_of_hearts.rank

'Q'

In [23]:
queen_of_hearts

NamedTupleCard(rank='Q', suit='Hearts')

In [24]:
queen_of_hearts == NamedTupleCard('Q', 'Hearts')

True

By design, a `namedtuple` is a **regular tuple**. This can be seen in comparisons, for instance:

In [25]:
queen_of_hearts == ('Q', 'Hearts')

True

While this might seem like a good thing, this **lack of awareness** about its **own type** can lead to subtle and hard-to-find bugs, especially since it will also happily compare two different `namedtuple` classes:

In [27]:
Person = namedtuple('Person', ['first_initial', 'last_name'])

In [28]:
ace_of_spades = NamedTupleCard('A', 'Spades')

In [29]:
ace_of_spades == Person('A', 'Spades')

True

The `namedtuple` also comes with some restrictions. For instance, it is hard to add default values to some of the fields in a `namedtuple`. A `namedtuple` is also by nature immutable. That is, the value of a `namedtuple` can never change. 

In [30]:
card = NamedTupleCard('7', 'Diamonds')
card.rank = '9'

AttributeError: can't set attribute

### Basic Data Classes

In [31]:
from dataclasses import dataclass

@dataclass
class Position:
    name: str
    lon: float
    lat: float

In [33]:
pos = Position('Oslo', 10.8, 59.9)
pos

Position(name='Oslo', lon=10.8, lat=59.9)

In [34]:
pos.lat

59.9

In [35]:
print(f'{pos.name} is at {pos.lat}°N, {pos.lon}°E')

Oslo is at 59.9°N, 10.8°E


`from dataclasses import make_dataclass`

You can also create data classes **similarly to how named tuples are created**. The following is (almost) equivalent to the definition of Position above:

In [36]:
from dataclasses import make_dataclass

Position = make_dataclass('Position', ['name', 'lat', 'lon'])

### Default Values
It is easy to add default values to the fields of your data class:

In [37]:
from dataclasses import dataclass

@dataclass
class Position:
    name: str
    lon: float = 0.0
    lat: float = 0.0

This works exactly as if you had specified the default values in the definition of the `.__init__()` method of a regular class:

In [38]:
Position('Null Island')

Position(name='Null Island', lon=0.0, lat=0.0)

In [39]:
Position('Greenwich', lat=51.8)

Position(name='Greenwich', lon=0.0, lat=51.8)

In [40]:
Position('Vancouver', -123.1, 49.3)

Position(name='Vancouver', lon=-123.1, lat=49.3)

### Type Hints
In fact, adding **some kind** of type hint is **mandatory** when defining the fields in your data class. Without a type hint, the field will not be a part of the data class. However, if you do not want to add explicit types to your data class, use `typing.Any`:

In [41]:
from dataclasses import dataclass
from typing import Any

@dataclass
class WithoutExplicitTypes:
    name: Any
    value: Any = 42

In [42]:
Position(3.14, 'pi day', 2018)

Position(name=3.14, lon='pi day', lat=2018)

This is how typing in Python usually works: Python is and will always be a dynamically typed language. To actually catch type errors, type checkers like Mypy can be run on your source code.

### Adding Methods
You can add a `.distance_to()` method to your data class just like you can with normal classes:

In [43]:
from dataclasses import dataclass
from math import asin, cos, radians, sin, sqrt

@dataclass
class Position:
    name: str
    lon: float = 0.0
    lat: float = 0.0

    def distance_to(self, other):
        r = 6371  # Earth radius in kilometers
        lam_1, lam_2 = radians(self.lon), radians(other.lon)
        phi_1, phi_2 = radians(self.lat), radians(other.lat)
        h = (sin((phi_2 - phi_1) / 2)**2
             + cos(phi_1) * cos(phi_2) * sin((lam_2 - lam_1) / 2)**2)
        return 2 * r * asin(sqrt(h))

It works as you would expect:

In [44]:
oslo = Position('Oslo', 10.8, 59.9)

In [45]:
vancouver = Position('Vancouver', -123.1, 49.3)

In [46]:
oslo.distance_to(vancouver)

7181.784122942117

### Advanced Default Values
The `field()` specifier is used to customize each field of a data class individually. 

Parameters `field()` supports:
- `default`: Default value of the field
- `default_factory`: Function that returns the initial value of the field
- `init`: Use field in `.__init__()` method? (Default is `True`.)
- `repr`: Use field in `repr` of the object? (Default is `True`.)
- `compare`: Include the field in comparisons? (Default is `True`.)
- `hash`: Include the field when calculating `hash()`? (Default is to use the same as for compare.)
- `metadata`: A mapping with information about the field

The `metadata` parameter is not used by the data classes themselves but is available for you (or third party packages) to attach information to fields. In the `Position` example, you could for instance specify that latitude and longitude should be given in degrees:

In [47]:
from dataclasses import dataclass, field

@dataclass
class Position:
    name: str
    lon: float = field(default=0.0, metadata={'unit': 'degrees'})
    lat: float = field(default=0.0, metadata={'unit': 'degrees'})

The `metadata` (and other information about a field) can be retrieved using the `fields()` function (note the plural s):

In [48]:
from dataclasses import fields

fields(Position)

(Field(name='name',type=<class 'str'>,default=<dataclasses._MISSING_TYPE object at 0x000001BD29501D30>,default_factory=<dataclasses._MISSING_TYPE object at 0x000001BD29501D30>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=False,_field_type=_FIELD),
 Field(name='lon',type=<class 'float'>,default=0.0,default_factory=<dataclasses._MISSING_TYPE object at 0x000001BD29501D30>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({'unit': 'degrees'}),kw_only=False,_field_type=_FIELD),
 Field(name='lat',type=<class 'float'>,default=0.0,default_factory=<dataclasses._MISSING_TYPE object at 0x000001BD29501D30>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({'unit': 'degrees'}),kw_only=False,_field_type=_FIELD))

In [49]:
lat_unit = fields(Position)[2].metadata['unit']
lat_unit

'degrees'

### Immutable Data Classes
One of the defining features of the `namedtuple` you saw earlier is that it is **immutable**. That is, the value of its **fields may never change**. For many types of data classes, this is a great idea! To make a data class immutable, set `frozen=True` when you create it. For example, the following is an immutable version of the `Position` class:

In [50]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Position:
    name: str
    lon: float = 0.0
    lat: float = 0.0

In a **frozen** data class, you can **not** assign values to the fields after creation:

In [51]:
pos = Position('Oslo', 10.8, 59.9)
pos.name

'Oslo'

In [52]:
pos.name = 'Stockholm'

FrozenInstanceError: cannot assign to field 'name'

In [55]:
pos.lon = 0.0

FrozenInstanceError: cannot assign to field 'lon'

Be aware though that if your data class contains mutable fields, those might still change. This is true for all nested data structures in Python

### Inheritance

In [56]:
from dataclasses import dataclass

@dataclass
class Position:
    name: str
    lon: float
    lat: float

@dataclass
class Capital(Position):
    country: str

In [57]:
Capital('Oslo', 10.8, 59.9, 'Norway')

Capital(name='Oslo', lon=10.8, lat=59.9, country='Norway')

The `country` field of Capital is added after the three original fields in `Position`. Things get a little more complicated if any fields in the base class **have default** values:

In [58]:
from dataclasses import dataclass

@dataclass
class Position:
    name: str
    lon: float = 0.0
    lat: float = 0.0

@dataclass
class Capital(Position):
    country: str  # Does NOT work

TypeError: non-default argument 'country' follows default argument 'lat'

The problem is that our new `country` field **has no default value**, while the `lon` and `lat` fields have default values. The data class will try to write an `.__init__()` method with the following signature:

In [ ]:
def __init__(name: str, lon: float = 0.0, lat: float = 0.0, country: str):
    ...

However, this is not valid Python. If a parameter has a default value, all following parameters must also have a default value. In other words, if a field in a base class has a default value, then all new fields added in a subclass must have default values as well.

Another thing to be aware of is how fields are ordered in a subclass. Starting with the base class, fields are ordered in the order in which they are first defined. If a field is redefined in a subclass, its order does not change. For example, if you define `Position` and `Capital` as follows:

In [59]:
from dataclasses import dataclass

@dataclass
class Position:
    name: str
    lon: float = 0.0
    lat: float = 0.0

@dataclass
class Capital(Position):
    country: str = 'Unknown'
    lat: float = 40.0

Then the order of the fields in `Capital` will still be `name`, `lon`, `lat`, `country`. However, the default value of `lat` will be `40.0`.

In [60]:
Capital('Madrid', country='Spain')

Capital(name='Madrid', lon=0.0, lat=40.0, country='Spain')

### Optimizing Data Classes

Slots can be used to make classes faster and use less memory. Data classes have no explicit syntax for working with slots, but the normal way of creating slots works for data classes as well.

In [61]:
from dataclasses import dataclass

@dataclass
class SimplePosition:
    name: str
    lon: float
    lat: float

@dataclass
class SlotPosition:
    __slots__ = ['name', 'lon', 'lat']
    name: str
    lon: float
    lat: float

Essentially, slots are defined using `.__slots__` to list the variables on a class. Variables or attributes not present in `.__slots__` may not be defined. Furthermore, a slots class may not have default values. The slot class is about 35% faster.